# Testing `fairscape_models.sql`

In [1]:
#%pip install pydantic sqlalchemy

## Setup 

In [2]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [3]:
try:
	os.remove("integration_test.db")
except:
	pass

In [4]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

from fairscape_models.utils import readCrate
from fairscape_models.sql.models import *
from fairscape_models.sql.ingest import ROCrateIngestRequest
import sqlalchemy as sa
import pathlib

## Load ROCrate Tests

In [5]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [6]:
len(test_rocrate.metadataGraph)

20552

In [7]:
# TODO create a flamegraph of loading rocrate
# py-spy
# pip install py-spy
# py-spy record -o profile.svg -- python myscript.py 
#
# flameprof for cProfile stats
# python -m cProfile -o script.prof myscript.py
# 

In [8]:
# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")


# create table 
Base.metadata.create_all(engine)



In [9]:
# create a session
session = sa.orm.Session(engine) 

ingestRequest = ROCrateIngestRequest(
	model=test_rocrate,
	session=session,
)

In [10]:
# test digest authors
author_data = ingestRequest._digest_authors()



In [11]:
# check authors
author_data, existing_author_ids = ingestRequest._check_authors(author_data)

# if no id's are found author id is an empty dictionary
existing_author_ids

{}

In [12]:
# write the authors
author_ids = ingestRequest._write_authors(author_data)

# join existing author ids to author_ids
full_author_ids = author_ids | existing_author_ids

In [13]:
full_author_ids

{'Nicole M. Mattson': 1,
 'Ishan Gaur': 2,
 'Jing Chen': 3,
 'Abantika Pal': 4,
 'William Leineweber': 5,
 'Ignacia Echeverria': 6,
 'Robin Bachelder': 7,
 'Leonard J. Foster': 8,
 'Keiichiro Ono': 9,
 'Ernst Pulido': 10,
 'Neelesh Soni': 11,
 'Xiaoyu Zhao': 12,
 'Katherine Licon': 13,
 'Joanna Lenkiewicz': 14,
 'Mengzhou Hu': 15,
 'Gege Qian': 16,
 'J. Wade Harper': 17,
 'Trey Ideker': 18,
 'Peter Zage': 19,
 'Dorothy Tsai': 20,
 'Emma Lundberg ': 21,
 'Laura Pontano Vaites': 22,
 'Kyung-Mee Moon': 23,
 'Edward L. Huttlin': 24,
 'Steven P. Gygi': 25,
 'Anthony Cesnik': 26,
 'Dexter Pratt': 27,
 'Leah V. Schaffer': 28,
 'Trang Le': 29,
 'Aji Palar': 30,
 'Andrej Sali': 31,
 'Yue Qin': 32,
 'Andrew P. Latham': 33,
 'Christopher Churas': 34}

In [14]:
author_ids

{'Nicole M. Mattson': 1,
 'Ishan Gaur': 2,
 'Jing Chen': 3,
 'Abantika Pal': 4,
 'William Leineweber': 5,
 'Ignacia Echeverria': 6,
 'Robin Bachelder': 7,
 'Leonard J. Foster': 8,
 'Keiichiro Ono': 9,
 'Ernst Pulido': 10,
 'Neelesh Soni': 11,
 'Xiaoyu Zhao': 12,
 'Katherine Licon': 13,
 'Joanna Lenkiewicz': 14,
 'Mengzhou Hu': 15,
 'Gege Qian': 16,
 'J. Wade Harper': 17,
 'Trey Ideker': 18,
 'Peter Zage': 19,
 'Dorothy Tsai': 20,
 'Emma Lundberg ': 21,
 'Laura Pontano Vaites': 22,
 'Kyung-Mee Moon': 23,
 'Edward L. Huttlin': 24,
 'Steven P. Gygi': 25,
 'Anthony Cesnik': 26,
 'Dexter Pratt': 27,
 'Leah V. Schaffer': 28,
 'Trang Le': 29,
 'Aji Palar': 30,
 'Andrej Sali': 31,
 'Yue Qin': 32,
 'Andrew P. Latham': 33,
 'Christopher Churas': 34}

In [15]:
# TODO slow?
ingestRequest._write_identifier_authors(full_author_ids)

In [16]:
# check that author table has correct author information
session.scalar(sa.select(sa.func.count(AuthorSQL.id)))

34

In [17]:
# check linked authors
session.scalar(sa.select(sa.func.count(AuthorIdentifierSQL.id)))

698633

In [18]:
# digest identifiers
rocrate_identifiers = ingestRequest._digest_identifiers()
ingestRequest._write_identifiers(rocrate_identifiers)

session.flush()
session.commit()

In [19]:
# check the identifiers

In [20]:
# digest all crate elements 
crate_elements = ingestRequest._digest_iterate_elements()

In [21]:
# write rocrate elements
ingestRequest._write_elements()

In [22]:
session.commit()

In [23]:
session.close()

## Test Query

In [24]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [25]:
from fairscape_models.sql.query import QueryByGUID, QueryResponse, SERIALIZE_TYPE
from fairscape_models.utils import readCrate
import sqlalchemy as sa
import pathlib

from fairscape_models.dataset import Dataset
from fairscape_models.software import Software
from fairscape_models.computation import Computation
from fairscape_models.rocrate import ROCrateMetadataElem

# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")

# create table 
# Base.metadata.create_all(engine)


In [26]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [27]:
test_crate_metadata = test_rocrate.getCrateMetadata()
test_crate_software = test_rocrate.getSoftware()[0]
test_crate_computation = test_rocrate.getComputations()[0]
test_crate_dataset = test_rocrate.getDatasets()[0]

In [28]:
test_crate_dataset.fileFormat

'.tsv'

In [29]:
session = sa.orm.Session(engine)

In [30]:
def runQuery(GUID: str, session):
	query = QueryByGUID(GUID)
	results = query.execute(session)
	
	return results.transform()


In [31]:
dataset_result = runQuery(test_crate_dataset.guid, session)

#dataset_query = QueryByGUID(test_crate_dataset.guid)
#dataset_query_response = dataset_query.execute(session)
#dataset_query_response.transform()

In [32]:
computation_result = runQuery(test_crate_computation.guid, session)

In [33]:
computation_query = QueryByGUID(test_crate_computation.guid)
computation_query_response = computation_query.execute(session)

In [34]:
computation_query_response.rootEntity

In [35]:
computation_query_response._transform_root_entity()

In [36]:
computation_query_response._convert_metadata()

In [37]:
computation_query_response.metadata

{'dateCreated': '2023-08-01',
 'name': 'Image Download from Human Protein Atlas',
 'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 'usedSoftware': [{'@id': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader'}],
 'datePublished': None,
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [38]:
from fairscape_models.sql.conversion.construct import ConvertComputationToSQL


test_comp = ConvertComputationToSQL(test_crate_computation)

In [39]:
computation_query_response.metadata

{'dateCreated': '2023-08-01',
 'name': 'Image Download from Human Protein Atlas',
 'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 'usedSoftware': [{'@id': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader'}],
 'datePublished': None,
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [40]:
software_result = runQuery(test_crate_software.guid, session)

In [41]:
crate_result = runQuery(test_crate_metadata.guid, session)

In [42]:
test_crate_computation.metadataType

['prov:Activity', 'https://w3id.org/EVI#Computation']

In [43]:
type(computation_result)

fairscape_models.computation.Computation

## Full ROCrate Query

In [49]:
test_rocrate_guid = test_crate_metadata.guid

# query identifier metadata
#type_query = sa.select(IdentifiersSQL.metadataType).where(IdentifiersSQL.guid == test_rocrate_guid)

# membership query
children_guid_query = sa.select(MembershipSQL.childGUID, MembershipSQL.childType).where(MembershipSQL.parentGUID == test_rocrate_guid)

children_guid_results = session.execute(children_guid_query).all()

In [54]:
test_rocrate_guid

'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'

In [50]:
children_guid_results

[('ark:59853/software-cellmaps-imagedownloader', <MetadataTypeEnumSQL.SOFTWARE: 'SOFTWARE'>),
 ('ark:59853/dataset-image-gene-node-attributes-with-locations', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-gene-node-attributes', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/computation-image-download', <MetadataTypeEnumSQL.COMPUTATION: 'COMPUTATION'>),
 ('ark:59853/dataset-image-1174_c5_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1913_h4_2', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1521_a3_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-778_h1_4', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1193_d6_5', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1235_g2_2', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-418_h11_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1191_d7_1', 

In [51]:
element_guid, element_type = children_guid_results[0]

In [52]:
from fairscape_models.sql.query import TYPE_LOOKUP, METADATA_TYPE_PROPERTY

metadata = {}
root_entity_query = sa.select(TYPE_LOOKUP[element_type]).filter_by(guid=element_guid)
root_entity_results = session.scalars(root_entity_query).all()

# keywords
keyword_query = sa.select(KeywordSQL.keywordValue).filter_by(guid=element_guid)

# authors
author_identifier_query = sa.select(AuthorSQL.name, AuthorSQL.orcid).join(AuthorIdentifierSQL, AuthorIdentifierSQL.author_id == AuthorSQL.id).where(AuthorIdentifierSQL.identifier_guid==element_guid)

author_results = session.execute(author_identifier_query).all()

# hasPart
has_part_query = sa.select(MembershipSQL.childGUID).where(MembershipSQL.parentGUID==element_guid)
has_part_results = session.execute(has_part_query).all()

# isPartOf
is_part_of_query = sa.select(MembershipSQL.parentGUID).where(MembershipSQL.childGUID==element_guid)
is_part_of_results = session.execute(is_part_of_query).all()

match element_type:
	case MetadataTypeEnumSQL.DATASET:
		# if its a dataset usedBy/generatedBy
		used_by_query = sa.select(ComputationUsedDatasetSQL.computationGUID).where(ComputationUsedDatasetSQL.datasetGUID == element_guid)
		used_by_results = session.execute(used_by_query).all()


		generated_by_query = sa.select(ComputationGeneratedDatasetSQL.computationGUID).where(ComputationUsedDatasetSQL.datasetGUID == element_guid)
		generated_by_results = session.execute(generated_by_query).all()

		pass
	case MetadataTypeEnumSQL.SOFTWARE:
		# if its a software usedBy
		used_by_query = sa.select(ComputationSQL.guid).where(ComputationSQL.usedSoftware == element_guid)
		used_by_results = session.execute(used_by_query).all()

		pass
	case MetadataTypeEnumSQL.COMPUTATION:
	# if its a computation used/generated
		used_query = sa.select(ComputationUsedDatasetSQL.datasetGUID).where(ComputationUsedDatasetSQL.computationGUID == element_guid)
		used_results = session.execute(used_query).all()

		generated_query = sa.select(ComputationGeneratedDatasetSQL.datasetGUID).where(ComputationGeneratedDatasetSQL.computationGUID == element_guid)
		generated_results = session.execute(generated_query).all()



In [53]:
metadata

{}